# <center> **<span style="font-size:80px;">Join: Barrios x Municipios</span>** </center>

Merge ponderado entre los datos **AMAEM** (nivel barrio) y los datos **INE** de municipios (plazas VT y porcentaje VT), usando el mapa de pesos definido en `src/config/barrio_mapping.py`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import sys
import os


from pathlib import Path

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    confusion_matrix, classification_report
)


sys.path.append(os.path.abspath(os.path.join("..")))
from src.config import DatasetKeys
from src.config import Paths
Paths.init_project()

In [ ]:
from src.water2fraud.features.preprocessor import WaterPreprocessor
preprocessor = WaterPreprocessor()

In [ ]:
seed = 80

images_path = Path("./EXTERNAL")
images_path.mkdir(parents=True, exist_ok=True)

In [ ]:
df_amaem = pd.read_csv(Paths.PROC_CSV_AMAEM)
df_amaem[DatasetKeys.FECHA] = pd.to_datetime(df_amaem[DatasetKeys.FECHA])

In [ ]:
def process_df(df, mapeo=None, drop=None):
    df.columns = (df.columns
                  .str.strip().str.lower()
                  .str.replace(' ', '_').str.replace(':', '')
                  .str.replace('á', 'a').str.replace('é', 'e')
                  .str.replace('í', 'i').str.replace('ó', 'o')
                  .str.replace('ú', 'u'))
    
   
    if mapeo:
        df = df.rename(columns=mapeo)
    
    if drop:
        df = df.drop(columns=drop)
        
    return df

In [ ]:
params_mun = {"encoding": "latin1", "sep": "\t"}

# Cargar archivos de Municipios
df_municipios_plazas   = pd.read_csv(Paths.INE_MUNICIPIOS_PLAZAS, **params_mun)
df_municipios_porcent  = pd.read_csv(Paths.INE_MUNICIPIOS_PORCENT, **params_mun)

### Merge Ponderado Barrio → Municipio (INE)

In [ ]:
from src.config import BARRIO_MUNICIPIO_WEIGHTS

# ── 1. Normalizar df_municipios_plazas y df_municipios_porcent ───────────────
# Extraemos el código INE de 5 dígitos (ej. '03014 Alacant/Alicante' → '03014')
df_mun_plazas  = process_df(df_municipios_plazas)
df_mun_porcent = process_df(df_municipios_porcent)

for df_mun in [df_mun_plazas, df_mun_porcent]:
    df_mun['cod_ine']   = df_mun['municipios'].str.extract(r'^(\d{5})')
    df_mun['mes_union'] = pd.to_datetime(
        df_mun['periodo'].str.replace('M', '-'), format='%Y-%m'
    ).dt.to_period('M')

print('Municipios en plazas:', df_mun_plazas['cod_ine'].nunique())
print('Municipios en porcentaje:', df_mun_porcent['cod_ine'].nunique())

In [ ]:
# ── 2. Construir tabla de pesos barrio → municipio ────────────────────────────
# El mapping ahora usa strings completos: '03014 Alacant/Alicante'
# Extraemos el cod_ine de 5 dígitos con regex para el join (igual que df_mun_*)
rows = []
for barrio, pesos in BARRIO_MUNICIPIO_WEIGHTS.items():
    for municipio_str, peso in pesos:
        rows.append({'barrio': barrio, 'municipio_nombre': municipio_str, 'peso': peso})

df_pesos = pd.DataFrame(rows, columns=['barrio', 'municipio_nombre', 'peso'])
df_pesos['cod_ine'] = df_pesos['municipio_nombre'].str.extract(r'^(\d{5})')

sin_datos = set(BARRIO_MUNICIPIO_WEIGHTS) - set(df_pesos['barrio'])
print(f'Barrios con algún municipio asignado : {df_pesos["barrio"].nunique()}')
print(f'Barrios sin datos INE (0%)           : {sin_datos}')
df_pesos[['barrio', 'municipio_nombre', 'cod_ine', 'peso']].drop_duplicates('barrio').head(57)

In [ ]:
# ── 3. Merge ponderado con plazas VT y porcentaje VT ─────────────────────────
cols_excluir = {'municipios', 'periodo', 'mes_union', 'cod_ine'}
cols_plazas  = [c for c in df_mun_plazas.columns  if c not in cols_excluir]
cols_porcent = [c for c in df_mun_porcent.columns if c not in cols_excluir]

# (a) Añadir mes_union al AMAEM y unir pesos
df_ext = df_amaem.copy()
df_ext['mes_union'] = df_ext[DatasetKeys.FECHA].dt.to_period('M')
df_ext = df_ext.merge(df_pesos, on='barrio', how='left')

def merge_ponderado(df_base, df_mun, cols_mun, sufijo):
    """Hace merge con df_mun, aplica el peso y agrega por barrio+fecha."""
    df = df_base.merge(
        df_mun[['cod_ine', 'mes_union'] + cols_mun],
        on=['cod_ine', 'mes_union'], how='left'
    )
    num_cols = df_mun[cols_mun].select_dtypes('number').columns.tolist()
    for col in num_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce') * df['peso']

    group_cols = list(df_amaem.columns) + ['mes_union']
    df_agg = (
        df.groupby(group_cols, observed=True)[num_cols]
          .sum(min_count=1)
          .reset_index()
    )
    df_agg = df_agg.rename(columns={c: f'{sufijo}_{c}' for c in num_cols})
    return df_agg

df_plazas_resultado  = merge_ponderado(df_ext, df_mun_plazas,  cols_plazas,  'plazas')
df_porcent_resultado = merge_ponderado(df_ext, df_mun_porcent, cols_porcent, 'porcent')

# (b) Unir ambos resultados
join_on = list(df_amaem.columns) + ['mes_union']
df_municipios_merged = df_plazas_resultado.merge(df_porcent_resultado, on=join_on, how='left')

print('Shape final:', df_municipios_merged.shape)
nuevas = [c for c in df_municipios_merged.columns if c.startswith(('plazas_', 'porcent_'))]
print('Nuevas columnas:', nuevas)

In [ ]:
# ── 4. Resumen de cobertura ───────────────────────────────────────────────────
nuevas_cols = [c for c in df_municipios_merged.columns if c.startswith(('plazas_', 'porcent_'))]
if nuevas_cols:
    nan_pct = df_municipios_merged[nuevas_cols[0]].isna().mean() * 100
    print(f'Filas con NaN en datos municipio: {nan_pct:.1f}%')

barrios_sin_cobertura = [b for b, p in BARRIO_MUNICIPIO_WEIGHTS.items() if not p]
print(f'Barrios sin cobertura INE: {barrios_sin_cobertura}')
df_municipios_merged.drop(columns=['mes_union']).head(5)

## Provincia (VT + Hoteles)

In [ ]:
params_prov = {"encoding": "utf-8", "sep": ";"}

# Cargar archivos de Provincia
df_provincia_hotel = pd.read_csv(Paths.INE_PROVINCIA_HOTEL, **params_prov)
df_provincia_vt    = pd.read_csv(Paths.INE_PROVINCIA_VT,    **params_prov)

In [ ]:
# Normalizar columnas
mapeo_prov = {
    "Fecha": DatasetKeys.FECHA,
    "total_numero_de_alojamientos_turisticos_ocupados": "ocupaciones_vt",
    "numero_de_noches_ocupadas": "pernoctaciones_vt",
    "estancia_media_alacant/alicante": "media_vt"
}

df_provincia_vt = process_df(df_provincia_vt, mapeo=mapeo_prov)

# Convertir periodo 'AñoMMes' → Period mensual
# Esto hace que '2024-06-30' (AMAEM) y '2024-06-01' (INE) sean ambos '2024-06'
df_provincia_vt['mes_union'] = pd.to_datetime(
    df_provincia_vt[DatasetKeys.FECHA].str.replace('M', '-')
).dt.to_period('M')

# Columnas a mergear de provincia VT (excluir fecha y mes_union)
cols_prov_vt = [c for c in df_provincia_vt.columns
                if c not in (DatasetKeys.FECHA, 'mes_union')]
print('Columnas VT provincia:', cols_prov_vt)

In [ ]:
# ── Merge provincia VT con df_municipios_merged ───────────────────────────────
# df_municipios_merged ya tiene 'mes_union' de la celda anterior
# Si no tuviese mes_union, la añadimos:
if 'mes_union' not in df_municipios_merged.columns:
    df_municipios_merged['mes_union'] = df_municipios_merged[DatasetKeys.FECHA].dt.to_period('M')

df_resultado = pd.merge(
    df_municipios_merged,
    df_provincia_vt[['mes_union'] + cols_prov_vt],  # solo columnas de turismo
    on='mes_union',
    how='left'   # left: no se pierde ninguna fila de AMAEM
)

df_resultado = df_resultado.drop(columns=['mes_union'])

print('Shape df_resultado:', df_resultado.shape)
print('Columnas nuevas de provincia:', [c for c in df_resultado.columns if c in cols_prov_vt])

In [ ]:
# Mostrar muestra del resultado final
display(df_resultado.drop_duplicates(subset=[DatasetKeys.FECHA]).head(15))

## Comunidad Valenciana

In [ ]:
params_com = {"encoding": "latin1", "sep": ";"}

# Cargar archivos de Comunidad
df_comunidad_tipo_aloj = pd.read_csv(Paths.INE_COMUNIDAD_TIPO_ALOJ, **params_com)
df_comunidad_total     = pd.read_csv(Paths.INE_COMUNIDAD_TOTAL,     **params_com)

In [ ]:
mapeo_comunidad = {
    'destino_principal': 'destino',
    'concepto_turistico': 'concepto',
    'alojamiento_principal_nivel_1': 'aloj_n1',
    'alojamiento_principal_nivel_2': 'aloj_n2',
    'motivo_principal': 'motivo',
    'transporte_principal': 'transporte',
    'forma_de_organizacion_del_viaje': 'organizacion',
    'duracion_del_viaje': 'duracion',
    'comunidad_autonoma_de_residencia': 'residencia',
    'periodo': 'periodo',
    'total': 'total'
}

drop_comunidad = ['destino', 'motivo', 'transporte', 'organizacion', 'duracion', 'residencia']

df_comunidad_tipo_aloj = process_df(df_comunidad_tipo_aloj, mapeo=mapeo_comunidad, drop=drop_comunidad)
df_comunidad_total     = process_df(df_comunidad_total,     mapeo=mapeo_comunidad, drop=drop_comunidad)

df_comunidad_total.head(5)

## Dataset Final Integrado

In [ ]:
print('=== df_resultado (dataset final integrado) ===')
print(f'Shape: {df_resultado.shape}')
print(f'Columnas ({len(df_resultado.columns)}):')
for col in df_resultado.columns:
    nan_pct = df_resultado[col].isna().mean() * 100
    print(f'  {col:<40} NaN={nan_pct:.1f}%')

### 4. Nuevas Variables Externas (GVA Alta Resolución)
Aquí integramos los nuevos datasets que tienen información temporal y geográfica exacta.

In [ ]:
# Cargar los nuevos CSVs generados con los scripts de preparación
import os
import pandas as pd

ruta_gva_vt = Paths.PROC_CSV_DIR / 'gva_municipios_vt.csv'
ruta_gva_hoteles = Paths.PROC_CSV_DIR / 'gva_municipios_hoteles.csv'

df_gva_vt = pd.read_csv(ruta_gva_vt, dtype={'cod_ine': str}) if os.path.exists(ruta_gva_vt) else pd.DataFrame()
df_gva_ho = pd.read_csv(ruta_gva_hoteles, dtype={'cod_ine': str}) if os.path.exists(ruta_gva_hoteles) else pd.DataFrame()

print(f"GVA VT: {len(df_gva_vt)} filas")
print(f"GVA Hoteles: {len(df_gva_ho)} filas")

In [ ]:
# ── 5. Integración Final de GVA Viviendas y Hoteles (Merge Ponderado) ──

# 1. Sincronizar mes_union como STRING para evitar fallos de matching (String vs Period)
if 'mes_union' not in df_resultado.columns:
    df_resultado['mes_union'] = pd.to_datetime(df_resultado['fecha']).dt.strftime('%Y-%m')
else:
    df_resultado['mes_union'] = df_resultado['mes_union'].astype(str)

# 2. Asegurarnos que df_pesos tenga cod_ine limpio
if 'cod_ine' not in df_pesos.columns:
    df_pesos['cod_ine'] = df_pesos['municipio_nombre'].str.extract(r'^(\\d{5})')

# 3. Función auxiliar para distribuir datos de municipio a barrio usando df_pesos normalizados
def distribuir_a_barrios(df_mun, valor_col, nombre_salida):
    # Calculamos la suma de pesos para cada municipio
    suma_pesos_mun = df_pesos.groupby('cod_ine')['peso'].sum().reset_index()
    suma_pesos_mun.rename(columns={'peso': 'peso_total_mun'}, inplace=True)
    df_pesos_norm = pd.merge(df_pesos[['barrio', 'cod_ine', 'peso']], suma_pesos_mun, on='cod_ine')
    df_pesos_norm['peso_norm'] = df_pesos_norm['peso'] / df_pesos_norm['peso_total_mun']
    
    # Cruzamos datos municipio con los pesos normalizados
    df_joined = pd.merge(df_mun, df_pesos_norm[['barrio', 'cod_ine', 'peso_norm']], on='cod_ine', how='inner')
    # Aplicamos el peso normalizado de forma que el total del municipio se divida entre sus barrios
    df_joined[nombre_salida] = df_joined[valor_col] * df_joined['peso_norm']
    # Agrupamos por mes y barrio (por si un barrio recibe de varios municipios)
    return df_joined.groupby(['mes_union', 'barrio'])[nombre_salida].sum().reset_index()

# 4. Procesar GVA VT
if not df_gva_vt.empty:
    df_gva_vt['mes_union'] = df_gva_vt['mes_union'].astype(str)
    df_vt_barrios = distribuir_a_barrios(df_gva_vt, 'plazas_gva_vt', 'plazas_vt_real')
    df_resultado = pd.merge(df_resultado, df_vt_barrios, on=['mes_union', 'barrio'], how='left')

# 5. Procesar GVA Hoteles
if not df_gva_ho.empty:
    df_gva_ho['mes_union'] = df_gva_ho['mes_union'].astype(str)
    df_ho_barrios = distribuir_a_barrios(df_gva_ho, 'plazas_gva_hoteles', 'plazas_hoteles_real')
    df_resultado = pd.merge(df_resultado, df_ho_barrios, on=['mes_union', 'barrio'], how='left')

print("Variables añadidas a df_resultado:")
print(df_resultado.columns.tolist()[-3:])
print(f"Muestra de valores unidos (no-NaNs): {df_resultado['plazas_vt_real'].notna().sum()} registros")